
# Lazy Evaluation, Lambda Calculus, and Church Numerals in Scala

During this lecture, we will cover:

- **Lazy Evaluation** — Separating Program Description from Evaluation
  - `lazy val` and call-by-name parameters (`=> A`)
  - Thunks: wrapping computations as values
  - `LazyList` — infinite and lazy collections
  - Practical use cases: short-circuiting, memoization, infinite streams
- **Lambda Calculus** — The Theory Underlying Functional Programming
  - Syntax: variables, abstraction, application
  - Beta-reduction and normal form
  - Encoding data and control flow with pure functions
- **Church Numerals** — Numbers and Arithmetic from Pure Lambda Calculus
  - Encoding natural numbers as higher-order functions
  - Successor, addition, multiplication, exponentiation
  - Boolean Church encoding
  - Converting Church numerals to Scala `Int`

These topics form the **theoretical and practical foundation** of functional programming. Lazy evaluation is used in every Haskell program, in Spark's execution model, and in Scala's own standard library. Lambda calculus is the minimal formal system from which all of computation can be derived — understanding it gives you insight into how every functional language actually works at its core.



# Part 1: Lazy Evaluation

## 1.1 Strict vs. Lazy Evaluation

In a **strictly evaluated** language (like most languages, including default Scala), every expression is evaluated **immediately** when it is encountered — before being passed to a function or bound to a name.

In a **lazily evaluated** language (like Haskell), expressions are evaluated **only when their value is actually needed** — and often **only once**, with the result cached (this is called *memoization* or *sharing*).

Scala is **strict by default** but provides several mechanisms to introduce laziness:

| Mechanism | What it does |
|-----------|-------------|
| `lazy val` | Delays evaluation of a `val` until first access; then memoizes |
| By-name parameter `=> A` | Re-evaluates the argument expression each time the parameter is used |
| `LazyList[A]` | A lazy, potentially infinite linked list with memoized elements |
| `() => A` (thunk) | A zero-argument function that defers any computation |


In [ ]:
// Strict evaluation — happens immediately
val strict = {
  println("[strict] Evaluating now!")
  42
}
// The println already ran above — before we ever use `strict`
println(s"strict = $strict")
println(s"strict again = $strict")  // no re-evaluation

In [ ]:
// lazy val — deferred until first use, then memoized
lazy val lazyVal = {
  println("[lazy val] Evaluating now!")
  42
}
// Nothing printed yet — the block has NOT run
println("Before first access")
println(s"lazyVal = $lazyVal")       // Triggers evaluation here
println(s"lazyVal again = $lazyVal") // Cached — no re-evaluation


## 1.2 `lazy val` — Deferred, Memoized Initialization

A `lazy val` is like a regular `val`, except:

1. Its right-hand side is **not evaluated** when the `lazy val` is declared
2. It is evaluated **the first time it is accessed**
3. The result is **stored** — subsequent accesses return the cached value without re-evaluating

This makes `lazy val` ideal for:
- **Expensive computations** that may never be needed
- **Circular dependencies** between values at initialization time
- **Module-level initialization** that should happen once, on demand


In [ ]:
// Example: expensive computation only paid if needed
def expensiveQuery(): Int = {
  println("[DB] Executing expensive query...")
  Thread.sleep(100) // simulate latency
  1_000_000
}

lazy val cachedResult = expensiveQuery()

def processIfNeeded(flag: Boolean): String =
  if (flag) s"Result: $cachedResult"
  else "Skipped"

println(processIfNeeded(false))  // query never runs
println(processIfNeeded(true))   // query runs here
println(processIfNeeded(true))   // cached — query does NOT run again

In [ ]:
// lazy val for breaking forward reference cycles
// Without lazy, this would cause a NullPointerException at startup:
lazy val a: Int = b + 1
lazy val b: Int = 10

println(s"a = $a")  // 11 — b is evaluated first when a is accessed
println(s"b = $b")  // 10


## 1.3 By-Name Parameters — Call-by-Name

A **by-name parameter** is declared with `=> A` instead of `A`. When a function takes a by-name parameter:

- The argument expression is **not evaluated before the call** — it is passed unevaluated
- The expression is **re-evaluated each time** the parameter is referenced inside the function body
- No result is cached (unlike `lazy val`)

By-name parameters let you build **control-flow abstractions** — functions that look like language keywords (`if`, `while`, `try`) — because you control *when* (and *whether*) the argument is evaluated.


In [ ]:
// Contrast: by-value vs. by-name

def byValue(x: Int): Int = {
  println(s"  Inside byValue, x = $x")
  x + x
}

def byName(x: => Int): Int = {
  println("  Inside byName, about to use x")
  val r = x + x   // x is re-evaluated TWICE
  r
}

println("Calling byValue:")
byValue({ println("  [arg] Evaluating argument"); 21 })
// argument is evaluated ONCE before the call

println("\nCalling byName:")
byName({ println("  [arg] Evaluating argument"); 21 })
// argument is evaluated TWICE — once per reference to x

In [ ]:
// By-name enables custom control flow: our own 'if'
def myIf[A](condition: Boolean, thenBranch: => A, elseBranch: => A): A =
  if (condition) thenBranch else elseBranch

// Only ONE branch is evaluated — exactly like the built-in if
val result = myIf(
  2 > 1,
  { println("  then branch evaluated"); "yes" },
  { println("  else branch evaluated"); "no" }  // never runs
)
println(s"result = $result")

In [ ]:
// By-name enables our own 'while'
def myWhile(condition: => Boolean)(body: => Unit): Unit =
  if (condition) {
    body
    myWhile(condition)(body)
  }

var count = 0
myWhile(count < 4) {
  println(s"  count = $count")
  count += 1
}
// condition and body are re-evaluated each iteration — just like a real while

In [ ]:
// By-name for safe resource handling
def withLogging[A](operationName: String)(block: => A): A = {
  println(s"[START] $operationName")
  val result = block  // block is a by-name parameter
  println(s"[END]   $operationName => $result")
  result
}

val x = withLogging("compute square") { 7 * 7 }
val y = withLogging("parse number")   { "42".toInt + 8 }

// By-name enables try-like semantics
def attempt[A](block: => A): Option[A] =
  try Some(block)
  catch { case _: Exception => None }

attempt("123".toInt)   // Some(123)
attempt("abc".toInt)   // None


## 1.4 Thunks — Wrapping Computation as a Value

A **thunk** is simply a zero-argument function `() => A` that wraps a computation. Calling the function forces the computation to run.

Thunks are the **explicit, first-class** version of by-name parameters: they can be stored in data structures, passed around, and called on demand — something by-name parameters cannot do.


In [ ]:
// A thunk is a zero-arg function
val thunk: () => Int = () => {
  println("  [thunk] Computing...")
  42
}

println("Thunk created — nothing computed yet")
println(s"First call:  ${thunk()}")   // computed
println(s"Second call: ${thunk()}")   // computed AGAIN — no memoization

// Thunks can be stored in collections
val tasks: List[() => String] = List(
  () => "task A",
  () => { println("  Running B"); "task B" },
  () => "task C"
)

// Execute only specific tasks
tasks(1)()  // run only task B
tasks.map(f => f())  // run all tasks

In [ ]:
// Building a lazy, memoized value manually (what lazy val does under the hood)
class Lazy[A](thunk: () => A) {
  private var evaluated: Boolean = false
  private var cache: A = _

  def force(): A =
    if (evaluated) {
      println("  [Lazy] Cache hit")
      cache
    } else {
      println("  [Lazy] Evaluating thunk")
      cache = thunk()
      evaluated = true
      cache
    }
}

val myLazy = new Lazy(() => {
  println("  [Lazy] Expensive work!")
  100 * 100
})

println("Before force")
println(s"Result: ${myLazy.force()}")
println(s"Result again: ${myLazy.force()}")


## 1.5 `LazyList` — Infinite and Lazy Collections

`LazyList[A]` (formerly `Stream[A]` in Scala 2.12 and earlier) is a **singly-linked list** where both the head and tail are computed **lazily**. Each element is computed only when it is accessed, and once computed, it is **memoized**.

This makes `LazyList` capable of representing **infinite sequences** — sequences that are conceptually unbounded but physically only compute as many elements as you actually consume.

The two constructors are:
- `LazyList.empty` — the empty lazy list
- `head #:: tail` — cons: prepend `head` to a lazily evaluated `tail`


In [ ]:
// Constructing a LazyList manually
val ll: LazyList[Int] = 1 #:: 2 #:: 3 #:: LazyList.empty

ll.head       // 1
ll.tail.head  // 2
ll.toList     // List(1, 2, 3)

// LazyList.from — infinite list of integers starting at n
val naturals: LazyList[Int] = LazyList.from(1)
// This is conceptually infinite — it has no end

naturals.take(10).toList   // List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
naturals.take(5).sum       // 15
naturals.take(100).last    // 100

In [ ]:
// Defining an infinite LazyList recursively
def from(n: Int): LazyList[Int] = n #:: from(n + 1)
// #:: is lazy on the right side — from(n+1) is NOT called until the tail is accessed

val nats = from(0)
nats.take(7).toList   // List(0, 1, 2, 3, 4, 5, 6)

// An infinite list of the same value
def repeat[A](x: A): LazyList[A] = x #:: repeat(x)

repeat("hello").take(4).toList   // List(hello, hello, hello, hello)

// An infinite list cycling through a finite list
def cycle[A](xs: List[A]): LazyList[A] = {
  def go(rest: List[A]): LazyList[A] = rest match {
    case Nil    => go(xs)
    case h :: t => h #:: go(t)
  }
  go(xs)
}

cycle(List(1, 2, 3)).take(10).toList   // List(1, 2, 3, 1, 2, 3, 1, 2, 3, 1)

In [ ]:
// The Sieve of Eratosthenes — infinite lazy prime stream
def sieve(s: LazyList[Int]): LazyList[Int] =
  s.head #:: sieve(s.tail.filter(_ % s.head != 0))

val primes: LazyList[Int] = sieve(LazyList.from(2))

primes.take(20).toList
// List(2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71)

// Find all primes below 100
primes.takeWhile(_ < 100).toList

// 100th prime
primes(99)

In [ ]:
// Fibonacci numbers — a classic lazy recursive definition
lazy val fibs: LazyList[BigInt] =
  BigInt(0) #:: BigInt(1) #:: fibs.zip(fibs.tail).map { case (a, b) => a + b }

fibs.take(15).toList
// List(0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377)

// The 50th Fibonacci number (BigInt handles large values)
fibs(50)

In [ ]:
// LazyList supports the same HOFs as List — lazily
val evens   = LazyList.from(0).filter(_ % 2 == 0)
val squares = LazyList.from(1).map(n => n * n)
val evenSquares = LazyList.from(1).map(n => n * n).filter(_ % 2 == 0)

evens.take(8).toList           // List(0, 2, 4, 6, 8, 10, 12, 14)
squares.take(8).toList         // List(1, 4, 9, 16, 25, 36, 49, 64)
evenSquares.take(5).toList     // List(4, 16, 36, 64, 100)

// Short-circuit: find the first element satisfying a predicate
// Even on an infinite list, this terminates immediately
LazyList.from(1).find(_ > 1000)   // Some(1001)


## 1.6 Practical Patterns: Short-Circuiting, Corecursion, Memoization

Lazy evaluation enables several important programming patterns that are impossible or awkward with strict evaluation.


In [ ]:
// Pattern 1: Short-circuit evaluation with by-name
// Scala's && and || already short-circuit, but you can build your own
def and(a: Boolean, b: => Boolean): Boolean = if (!a) false else b
def or(a: Boolean, b: => Boolean): Boolean  = if (a)  true  else b

// b is never evaluated if a is false
and(false, { println("  [b] evaluated!"); true })   // false, no println
or(true,   { println("  [b] evaluated!"); false })  // true, no println

// Pattern 2: Lazy logging — don't pay for message construction if log level is off
object Logger {
  var level = 1
  def debug(msg: => String): Unit = if (level >= 2) println(s"[DEBUG] $msg")
  def info(msg: => String): Unit  = if (level >= 1) println(s"[INFO]  $msg")
}

Logger.debug(s"Expensive string ${(1 to 1000).sum}")  // level 1, never built
Logger.info("Application started")                     // printed

In [ ]:
// Pattern 3: Corecursion — building data structures from a seed
// LazyList.unfold is the corecursive primitive

// unfold: given a state S and a function S => Option[(A, S)],
// produce a LazyList[A] by repeatedly applying the function
val countdown: LazyList[Int] = LazyList.unfold(10) { n =>
  if (n < 0) None
  else Some((n, n - 1))
}
countdown.toList  // List(10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0)

// Fibonacci via unfold (no recursion needed)
val fibsUnfold: LazyList[BigInt] =
  LazyList.unfold((BigInt(0), BigInt(1))) { case (a, b) =>
    Some((a, (b, a + b)))
  }
fibsUnfold.take(12).toList

// Powers of 2
val powersOf2: LazyList[Long] =
  LazyList.unfold(1L)(n => Some((n, n * 2)))
powersOf2.take(16).toList

In [ ]:
// Pattern 4: Lazy data pipelines — process only what you need
// Simulate reading a large log file
val logLines: LazyList[String] = LazyList(
  "INFO  2024-03-15 10:00:00 Server started",
  "DEBUG 2024-03-15 10:00:01 Connection pool initialized",
  "ERROR 2024-03-15 10:00:03 Failed to connect to DB: timeout",
  "INFO  2024-03-15 10:00:05 Retrying connection",
  "ERROR 2024-03-15 10:00:07 Failed again: connection refused",
  "INFO  2024-03-15 10:00:10 Fallback activated",
  "WARN  2024-03-15 10:00:12 Running in degraded mode"
)

// Find the first ERROR — stops as soon as found, never reads the rest
val firstError = logLines.find(_.startsWith("ERROR"))

// Collect all ERROR lines without materializing a full list first
val allErrors = logLines.filter(_.startsWith("ERROR")).toList

// Take the first 3 INFO lines — lazy: never processes ERROR or WARN lines unnecessarily
val firstThreeInfo = logLines.filter(_.startsWith("INFO")).take(3).toList

println(s"First error: $firstError")
println(s"All errors: $allErrors")
println(s"First 3 INFO: $firstThreeInfo")


---

# Part 2: Lambda Calculus

## 2.1 What Is Lambda Calculus?

**Lambda calculus** (λ-calculus) is a formal system invented by Alonzo Church in the 1930s. It is the **simplest possible programming language** — a language with only three constructs:

| Construct | Syntax | Meaning |
|-----------|--------|---------|
| Variable | `x` | A name that refers to a value |
| Abstraction | `λx. M` | A function with parameter `x` and body `M` |
| Application | `M N` | Apply function `M` to argument `N` |

That's it. No integers. No booleans. No conditionals. No recursion keyword. **Everything** can be encoded using only these three constructs and one reduction rule.

### Why does this matter?

Lambda calculus is **Turing complete** — it can compute everything a Turing machine can compute. Every functional programming language (Haskell, Scala, ML, Clojure, Lisp) is essentially an enriched lambda calculus with syntax sugar. Understanding lambda calculus gives you the **deep structural understanding** of why functional programs work the way they do.

### The One Reduction Rule: Beta-Reduction

The only way to compute in lambda calculus is **β-reduction** (beta reduction):

```
(λx. M) N  →β  M[x := N]
```

Read: *applying a function `λx. M` to an argument `N` produces the body `M` with every free occurrence of `x` replaced by `N`.*

A term with no further beta reductions possible is in **normal form** (the "result").


In [ ]:
// Lambda calculus in Scala: functions ARE lambda terms

// λx. x  — the identity function
val identity = (x: Any) => x

// λx. λy. x  — the first/const function (selects its first argument)
val const = (x: Any) => (y: Any) => x

// λx. λy. y  — the second function (selects its second argument)
val second = (x: Any) => (y: Any) => y

// Beta reduction manually:
// (λx. x) 42  →β  42
identity(42)              // 42

// (λx. λy. x) "hello" "world"  →β  (λy. "hello") "world"  →β  "hello"
const("hello")("world")   // "hello"

// (λx. λy. y) "hello" "world"  →β  "world"
second("hello")("world")  // "world"

In [ ]:
// Manually tracing beta-reduction

// Term: (λx. λy. x + y) 3 4
// Step 1: Apply to 3:  (λy. 3 + y) 4
// Step 2: Apply to 4:  3 + 4
// Normal form: 7

val add = (x: Int) => (y: Int) => x + y
add(3)(4)   // 7

// Term: (λf. λx. f (f x)) (λn. n + 1) 0
// Apply to (λn. n + 1):  (λx. (λn. n+1) ((λn. n+1) x)) 0
// Apply to 0:            (λn. n+1) ((λn. n+1) 0)
//                    →β  (λn. n+1) (0+1)
//                    →β  (λn. n+1) 1
//                    →β  1+1
//                    →β  2

val applyTwice = (f: Int => Int) => (x: Int) => f(f(x))
val increment  = (n: Int) => n + 1

applyTwice(increment)(0)    // 2
applyTwice(increment)(10)   // 12
applyTwice(_ * 2)(3)        // 12  (3 -> 6 -> 12)


## 2.2 Currying and Multi-Argument Functions

In pure lambda calculus, **all functions take exactly one argument**. Multi-argument functions are encoded via **currying** — a function that returns another function.

```
λx. λy. λz. body
```

is a function that takes `x`, and returns a function that takes `y`, and returns a function that takes `z`, and returns `body`.

This is exactly what Scala does with multiple parameter lists, and it was named after **Haskell Curry** — though the idea originated with Moses Schönfinkel.


In [ ]:
// Currying: a 3-argument function as nested lambdas
val f3 = (x: Int) => (y: Int) => (z: Int) => x + y + z

f3(1)(2)(3)    // 6
f3(10)(20)(30) // 60

// Partial application: fix some arguments, leave others open
val addTen  = f3(10)       // (y: Int) => (z: Int) => 10 + y + z
val add10_5 = f3(10)(5)    // (z: Int) => 15 + z

addTen(3)(7)    // 20
add10_5(100)    // 115

// Scala's built-in currying / uncurrying conversion
def sumCurried(x: Int)(y: Int): Int = x + y

val addFive = sumCurried(5)_  // partial application produces Int => Int
List(1, 2, 3, 4).map(addFive) // List(6, 7, 8, 9)


## 2.3 Combinators

A **combinator** is a lambda term with **no free variables** — it is a closed, self-contained expression. Combinators are the reusable building blocks of lambda calculus.

The most famous combinators:

| Name | Lambda | Scala | Meaning |
|------|--------|-------|---------|
| **I** | `λx. x` | `x => x` | Identity |
| **K** | `λx. λy. x` | `x => y => x` | Constant (always returns first arg) |
| **S** | `λx. λy. λz. x z (y z)` | complex | Substitution |
| **B** | `λf. λg. λx. f (g x)` | `f => g => x => f(g(x))` | Composition |
| **C** | `λf. λx. λy. f y x` | `f => x => y => f(y)(x)` | Flip arguments |

Remarkably, **S and K alone are Turing-complete** — every possible computation can be expressed using only these two combinators.


In [ ]:
// The classic combinators in Scala

// I combinator: λx. x
val I: Any => Any = x => x
I(42)       // 42
I("hello")  // "hello"

// K combinator: λx. λy. x  (const)
val K: Any => Any => Any = x => y => x
K(1)(2)     // 1
K("a")("b") // "a"

// B combinator: function composition  λf. λg. λx. f(g(x))
def B[A, B, C](f: B => C)(g: A => B)(x: A): C = f(g(x))

val double   = (x: Int) => x * 2
val addThree = (x: Int) => x + 3

val doubleThenAdd3 = B(addThree)(double)  // addThree(double(x))
doubleThenAdd3(5)   // 13  (5*2=10, 10+3=13)

// C combinator: flip  λf. λx. λy. f y x
def C[A, B, C](f: A => B => C)(x: B)(y: A): C = f(y)(x)

val subtract = (x: Int) => (y: Int) => x - y
val flipped  = C(subtract)  // now: y - x order
subtract(10)(3)   // 7
flipped(3)(10)    // 7  (arguments flipped)

In [ ]:
// The S combinator: λx. λy. λz. x z (y z)
// S applies x to z, and y to z, then applies the first result to the second
def S[A, B, C](x: A => B => C)(y: A => B)(z: A): C = x(z)(y(z))

// Interesting: S K K = I  (identity from just S and K)
// S K K z = K z (K z) = z
def SKK[A](z: A): A = S[A, A, A](K(_)(()))(K(_)(()))(z).asInstanceOf[A]
// (This is more conceptual — let's demonstrate the equivalence directly)

// S applied to two const-like functions recreates identity:
val result = S((x: Int) => (y: Int) => x)((x: Int) => 0)(42)
println(s"S K K 42 = $result")  // 42 — identity!

// S can also express function application:
// (λf. λx. f x) = S (K I)  -- but let's just show S in action
val applyF = S((f: Int => Int) => (x: Int) => f(x))((x: Int) => x + 0)
applyF(double)(5)   // 10


## 2.4 Church Booleans — Encoding True and False

In lambda calculus, we must encode **everything** — including booleans — as functions. The Church encoding of booleans is elegant:

- **`true`** = `λx. λy. x` — a function that takes two arguments and returns the first (K combinator)
- **`false`** = `λx. λy. y` — a function that takes two arguments and returns the second

Notice that a Church boolean **is** an `if-then-else` — it takes a `then` branch and an `else` branch and selects the correct one. This is the deep insight: in lambda calculus, **a boolean and a conditional are the same thing**.


In [ ]:
// Church Booleans
// A Church boolean is a function: Any => Any => Any
type ChurchBool = Any => Any => Any

val churchTrue:  ChurchBool = x => y => x  // λx. λy. x  (K combinator)
val churchFalse: ChurchBool = x => y => y  // λx. λy. y

// A Church boolean IS the if-then-else
// (churchBool)(thenBranch)(elseBranch) selects the right branch
def churchIf[A](cond: ChurchBool)(thenBranch: A)(elseBranch: A): A =
  cond(thenBranch)(elseBranch).asInstanceOf[A]

churchIf(churchTrue)("yes")("no")    // "yes"
churchIf(churchFalse)("yes")("no")   // "no"

In [ ]:
// Boolean operations from Church booleans

// NOT: λp. p false true
val churchNot: ChurchBool => ChurchBool =
  p => p(churchFalse)(churchTrue).asInstanceOf[ChurchBool]

// AND: λp. λq. p q false   (if p then q else false)
val churchAnd: ChurchBool => ChurchBool => ChurchBool =
  p => q => p(q)(churchFalse).asInstanceOf[ChurchBool]

// OR: λp. λq. p true q   (if p then true else q)
val churchOr: ChurchBool => ChurchBool => ChurchBool =
  p => q => p(churchTrue)(q).asInstanceOf[ChurchBool]

// Convert Church boolean to Scala Boolean for display
def toBool(cb: ChurchBool): Boolean =
  cb(true)(false).asInstanceOf[Boolean]

println(s"NOT true  = ${toBool(churchNot(churchTrue))}")
println(s"NOT false = ${toBool(churchNot(churchFalse))}")
println(s"true AND false = ${toBool(churchAnd(churchTrue)(churchFalse))}")
println(s"true AND true  = ${toBool(churchAnd(churchTrue)(churchTrue))}")
println(s"false OR true  = ${toBool(churchOr(churchFalse)(churchTrue))}")
println(s"false OR false = ${toBool(churchOr(churchFalse)(churchFalse))}")


---

# Part 3: Church Numerals — Numbers from Pure Functions

## 3.1 The Encoding: What Is a Number?

Church's most famous insight: **a natural number `n` can be represented as a function that applies another function `n` times**.

- **Zero** = `λf. λx. x` — apply `f` zero times to `x`
- **One** = `λf. λx. f x` — apply `f` once to `x`
- **Two** = `λf. λx. f (f x)` — apply `f` twice to `x`
- **Three** = `λf. λx. f (f (f x))` — apply `f` three times to `x`
- **n** = `λf. λx. f^n x` — apply `f` exactly `n` times to `x`

A Church numeral is a **higher-order function** that takes a function `f` and a starting value `x`, and applies `f` to `x` exactly `n` times. The numeral **encodes the concept of "doing something n times"**.

In Scala, the type of a Church numeral is:
```scala
type Church = [A] =>> (A => A) => A => A
```
or with a concrete type parameter:
```scala
type CN = (Int => Int) => Int => Int
```


In [ ]:
// Church numerals as Scala functions
// Type: (A => A) => A => A  for any A
// We'll use Int for concreteness

type CN = (Int => Int) => Int => Int

val zero:  CN = f => x => x
val one:   CN = f => x => f(x)
val two:   CN = f => x => f(f(x))
val three: CN = f => x => f(f(f(x)))
val four:  CN = f => x => f(f(f(f(x))))

// Convert a Church numeral to an Int:
// Apply (_ + 1) n times to 0 — counts the applications
def toInt(n: CN): Int = n(_ + 1)(0)

println(s"zero  = ${toInt(zero)}")
println(s"one   = ${toInt(one)}")
println(s"two   = ${toInt(two)}")
println(s"three = ${toInt(three)}")
println(s"four  = ${toInt(four)}")

In [ ]:
// Visualizing what Church numerals ARE
// A Church numeral n, given f and x, applies f to x exactly n times

def trace(label: String, n: CN): Unit = {
  var count = 0
  val step = (x: Int) => { count += 1; println(s"  f applied! count=$count"); x + 1 }
  val result = n(step)(0)
  println(s"$label: f was applied $count time(s), result = $result")
}

trace("zero ", zero)
trace("one  ", one)
trace("two  ", two)
trace("three", three)


## 3.2 Successor — Adding One

The **successor** function takes a Church numeral `n` and produces `n + 1`.

```
succ = λn. λf. λx. f (n f x)
```

Reading this: given `n`, produce a new numeral that applies `f` to `x` once more than `n` does.

- `n f x` applies `f` to `x` exactly `n` times
- `f (n f x)` applies `f` one additional time — giving `n + 1` applications total


In [ ]:
// Successor: λn. λf. λx. f(n f x)
val succ: CN => CN = n => f => x => f(n(f)(x))

val five  = succ(four)
val six   = succ(five)
val seven = succ(six)

println(s"succ(zero)  = ${toInt(succ(zero))}")
println(s"succ(one)   = ${toInt(succ(one))}")
println(s"succ(four)  = ${toInt(five)}")
println(s"succ(five)  = ${toInt(six)}")
println(s"succ(six)   = ${toInt(seven)}")

// Build any natural number by applying succ repeatedly
def churchFromInt(n: Int): CN =
  (0 until n).foldLeft(zero)((acc, _) => succ(acc))

(0 to 10).map(n => s"$n -> ${toInt(churchFromInt(n))}").foreach(println)


## 3.3 Addition

**Addition** of Church numerals has two elegant encodings:

**Method 1** — use `succ` repeatedly:
```
add = λm. λn. m succ n
```
Apply `succ` to `n` exactly `m` times. This is the definition of `m + n`.

**Method 2** — compose the application chains:
```
add = λm. λn. λf. λx. m f (n f x)
```
`n f x` applies `f` to `x` exactly `n` times, then `m f (...)` applies `f` to the result `m` more times — total `m + n` applications.


In [ ]:
// Addition Method 1: λm. λn. m succ n
// Apply succ m times starting from n
val add1: CN => CN => CN = m => n => m(succ)(n)

println(s"add1(two)(three) = ${toInt(add1(two)(three))}")  // 5
println(s"add1(zero)(four) = ${toInt(add1(zero)(four))}")  // 4

// Addition Method 2: λm. λn. λf. λx. m f (n f x)
val add2: CN => CN => CN = m => n => f => x => m(f)(n(f)(x))

println(s"add2(two)(three) = ${toInt(add2(two)(three))}")  // 5
println(s"add2(four)(four) = ${toInt(add2(four)(four))}")  // 8

// Alias for convenience
val add = add2

val eight = add(four)(four)
val ten   = add(four)(add(three)(three))
println(s"four + four = ${toInt(eight)}")
println(s"four + three + three = ${toInt(ten)}")


## 3.4 Multiplication

**Multiplication** is even more elegant:

```
mul = λm. λn. λf. m (n f)
```

Reading this: to apply `f` a total of `m × n` times, apply `(n f)` exactly `m` times. But `(n f)` is itself the operation "apply `f` exactly `n` times" — so applying it `m` times gives `m × n` total applications of `f`.

Alternatively:
```
mul = λm. λn. m (add n) zero
```
Apply `add n` to `zero` exactly `m` times — i.e., add `n` to itself `m` times.


In [ ]:
// Multiplication: λm. λn. λf. m(n f)
val mul: CN => CN => CN = m => n => f => m(n(f))

println(s"mul(two)(three)  = ${toInt(mul(two)(three))}")   // 6
println(s"mul(three)(four) = ${toInt(mul(three)(four))}")  // 12
println(s"mul(zero)(four)  = ${toInt(mul(zero)(four))}")   // 0
println(s"mul(one)(four)   = ${toInt(mul(one)(four))}")    // 4

// Alternative: m (add n) zero
val mul2: CN => CN => CN = m => n => m(add(n))(zero)

println(s"mul2(two)(three)  = ${toInt(mul2(two)(three))}")
println(s"mul2(three)(four) = ${toInt(mul2(three)(four))}")

val twelve    = mul(three)(four)
val twentyFour = mul(twelve)(two)
println(s"3 * 4 = ${toInt(twelve)}")
println(s"12 * 2 = ${toInt(twentyFour)}")


## 3.5 Exponentiation

**Exponentiation** — perhaps the most beautiful formula in Church numerals:

```
pow = λm. λn. n m
```

Yes — that is all. Apply `n` to `m`. Recall that `n` is a higher-order function that applies its first argument `n` times. So `n m` applies `m` (which means "multiply by m") to itself `n` times — giving `m^n`.

More precisely: `n m f = m^n f` — applying `f` a total of `m^n` times.


In [ ]:
// Exponentiation: λm. λn. n m
// m^n — apply m, n times
val pow: CN => CN => CN = m => n => n(m)

println(s"pow(two)(three)  = 2^3 = ${toInt(pow(two)(three))}")   // 8
println(s"pow(three)(two)  = 3^2 = ${toInt(pow(three)(two))}")   // 9
println(s"pow(two)(four)   = 2^4 = ${toInt(pow(two)(four))}")    // 16
println(s"pow(three)(three)= 3^3 = ${toInt(pow(three)(three))}") // 27
println(s"pow(two)(zero)   = 2^0 = ${toInt(pow(two)(zero))}")    // 1
println(s"pow(two)(one)    = 2^1 = ${toInt(pow(two)(one))}")     // 2

// Composition: (2^3)^2 = 64
val eightSquared = pow(pow(two)(three))(two)
println(s"(2^3)^2 = ${toInt(eightSquared)}")


## 3.6 Predecessor and Subtraction

The **predecessor** function (subtract 1) is the hardest operation to encode in Church numerals. The key insight is that we need to "count up to n-1" by building pairs.

The standard encoding uses a **pair trick**:
- Maintain a pair `(prev, curr)` initialized to `(0, 0)`
- At each step, advance the pair: `(prev, curr)` → `(curr, curr+1)`
- After `n` steps, `prev` holds `n-1`


In [ ]:
// Predecessor using the pair trick
// We'll implement it with Scala tuples to keep it readable
val pred: CN => CN = n => {
  // step: (prev, curr) -> (curr, succ(curr))
  val step: ((CN, CN)) => (CN, CN) = pair => (pair._2, succ(pair._2))
  // start: (zero, zero)
  // after n steps: (n-1, n)  — return the first element
  n(step)((zero, zero))._1
}

println(s"pred(one)   = ${toInt(pred(one))}")    // 0
println(s"pred(two)   = ${toInt(pred(two))}")    // 1
println(s"pred(three) = ${toInt(pred(three))}")  // 2
println(s"pred(four)  = ${toInt(pred(four))}")   // 3
println(s"pred(zero)  = ${toInt(pred(zero))}")   // 0 — floors at 0 (monus)

// Subtraction (monus): m - n  (floors at 0)
val sub: CN => CN => CN = m => n => n(pred)(m)
// Apply pred n times to m

println(s"sub(four)(two)   = ${toInt(sub(four)(two))}")   // 2
println(s"sub(three)(one)  = ${toInt(sub(three)(one))}")  // 2
println(s"sub(two)(four)   = ${toInt(sub(two)(four))}")   // 0 — floored


## 3.7 Predicates on Church Numerals

We can define **predicates** — tests that return Church booleans.

**`isZero`**: A Church numeral `n` applies its function `n` times. So `n (const false) true` is:
- If `n = 0`: applies `const false` zero times → returns `true`
- If `n > 0`: applies `const false` at least once → returns `false`


In [ ]:
// isZero: λn. n (λx. false) true
// Apply (_ => churchFalse) n times to churchTrue
// If n=0: returns churchTrue; if n>0: returns churchFalse
val isZero: CN => ChurchBool = n =>
  n.asInstanceOf[(ChurchBool => ChurchBool) => ChurchBool => ChurchBool]
   ((_: ChurchBool) => churchFalse)(churchTrue)

// Simpler approach with type casts for demonstration
def churchIsZero(n: CN): Boolean = {
  // Apply (_ => false) n times to true
  // If n=0: true unchanged; if n>0: false at first application
  val result = n.asInstanceOf[(Boolean => Boolean) => Boolean => Boolean]
    ((_: Boolean) => false)(true)
  result
}

println(s"isZero(zero)  = ${churchIsZero(zero)}")   // true
println(s"isZero(one)   = ${churchIsZero(one)}")    // false
println(s"isZero(two)   = ${churchIsZero(two)}")    // false
println(s"isZero(three) = ${churchIsZero(three)}")  // false


## 3.8 Putting It All Together: A Pure Lambda Calculator

Now let's build a complete demonstration that performs arithmetic using **only Church numerals** — no `Int` operations except for the final conversion to display results.


In [ ]:
// All arithmetic implemented purely in Church numerals
// No Int arithmetic is used in the computation — only in toInt for display

// Build numbers from scratch using only succ and zero
val c0 = zero
val c1 = succ(c0)
val c2 = succ(c1)
val c3 = succ(c2)
val c4 = succ(c3)
val c5 = succ(c4)
val c6 = succ(c5)
val c7 = succ(c6)
val c8 = succ(c7)
val c9 = succ(c8)
val c10 = succ(c9)

// Arithmetic demo — ALL computed via Church numeral operations
val results = List(
  ("2 + 3",      toInt(add(c2)(c3))),
  ("4 * 5",      toInt(mul(c4)(c5))),
  ("2 ^ 10",     toInt(pow(c2)(c10))),
  ("3 ^ 4",      toInt(pow(c3)(c4))),
  ("(2+3)*4",    toInt(mul(add(c2)(c3))(c4))),
  ("2^3 + 4*5",  toInt(add(pow(c2)(c3))(mul(c4)(c5)))),
  ("pred(9)",    toInt(pred(c9))),
  ("10 - 3",     toInt(sub(c10)(c3))),
  ("isZero(0)",  if (churchIsZero(c0)) 1 else 0),
  ("isZero(5)",  if (churchIsZero(c5)) 1 else 0)
)

println("=== Church Numeral Arithmetic ===")
results.foreach { case (expr, result) =>
  println(f"  $expr%-20s = $result")
}

In [ ]:
// Church numeral factorial — using recursion (Y combinator would be needed
// for true lambda calculus, but in Scala we can use a lazy val or def)

def churchFactorial(n: CN): CN =
  if (churchIsZero(n)) c1
  else mul(n)(churchFactorial(pred(n)))

println("=== Church Factorial ===")
val factInputs = List(c0, c1, c2, c3, c4, c5, c6, c7)
factInputs.zipWithIndex.foreach { case (n, i) =>
  println(f"  $i! = ${toInt(churchFactorial(n))}")
}

In [ ]:
// The Y Combinator — enabling recursion without naming
// In lambda calculus, functions have no names — recursion seems impossible.
// The Y combinator solves this: Y F = F (Y F)
// It finds the "fixed point" of F — a value x such that F(x) = x.

// Y = λf. (λx. f(x x))(λx. f(x x))
// In Scala, direct self-application causes infinite loops,
// so we use the lazy / by-name "Z combinator" variant:

// Z combinator (call-by-value Y): Z f = f (λv. Z f v)
def Z[A, B](f: (A => B) => (A => B)): A => B = {
  def selfApply(x: (A => B) => (A => B)): A => B =
    f(v => x(x)(v))
  selfApply(selfApply)
}

// Factorial without any explicit recursion — using Z combinator
val factZ: Int => Int = Z[Int, Int] { recurse =>
  n => if (n <= 0) 1 else n * recurse(n - 1)
}

println("=== Z Combinator Factorial (no named recursion!) ===")
(0 to 10).foreach(n => println(f"  $n! = ${factZ(n)}"))

// Fibonacci without explicit recursion
val fibZ: Int => Int = Z[Int, Int] { recurse =>
  n => if (n <= 1) n else recurse(n - 1) + recurse(n - 2)
}

println("\n=== Z Combinator Fibonacci ===")
(0 to 10).foreach(n => print(s"${fibZ(n)} "))
println()


## 3.9 Church Pairs — Encoding Data Structures

Beyond numbers and booleans, lambda calculus can encode **data structures**. A **Church pair** encodes a pair `(a, b)` as a function that applies a given selector to both elements:

```
pair = λa. λb. λf. f a b
fst  = λp. p (λa. λb. a)
snd  = λp. p (λa. λb. b)
```


In [ ]:
// Church Pairs
type CPair[A, B] = ((A, B) => Any) => Any

def churchPair[A, B](a: A)(b: B): CPair[A, B] =
  f => f(a, b)

def churchFst[A, B](p: CPair[A, B]): A =
  p((a, b) => a).asInstanceOf[A]

def churchSnd[A, B](p: CPair[A, B]): B =
  p((a, b) => b).asInstanceOf[B]

val pair42Hello = churchPair(42)("hello")
val pairBoolInt = churchPair(true)(99)

println(s"fst(42, 'hello') = ${churchFst(pair42Hello)}")
println(s"snd(42, 'hello') = ${churchSnd(pair42Hello)}")
println(s"fst(true, 99)   = ${churchFst(pairBoolInt)}")
println(s"snd(true, 99)   = ${churchSnd(pairBoolInt)}")

// Pairs enable the predecessor function's "slide" trick
// exactly as we implemented it above — (prev, curr) → (curr, succ(curr))


---

# Final Summary

## Lazy Evaluation

| Mechanism | Evaluates | Memoizes | Use Case |
|-----------|-----------|----------|-----------|
| `val` | Immediately | Yes (always fresh) | Default: eager binding |
| `lazy val` | On first access | Yes | Expensive or potentially unused values |
| `=> A` (by-name) | On each use | No | Control flow, custom `if`/`while` abstractions |
| `() => A` (thunk) | When called | No | First-class deferred computation |
| `LazyList` | Element by element | Yes (each element once) | Infinite sequences, lazy pipelines |

**Key principle**: Laziness separates **description** (what to compute) from **execution** (when to compute it). This enables infinite data structures, safe short-circuiting, and on-demand computation.

## Lambda Calculus

| Concept | Lambda Syntax | Scala Equivalent |
|---------|--------------|------------------|
| Variable | `x` | `x` |
| Abstraction | `λx. M` | `x => M` |
| Application | `M N` | `M(N)` |
| Beta reduction | `(λx. M) N →β M[x:=N]` | Function call |
| Currying | `λx. λy. M` | `x => y => M` |
| Identity (I) | `λx. x` | `x => x` |
| Const (K) | `λx. λy. x` | `x => y => x` |
| Compose (B) | `λf. λg. λx. f(g x)` | `f => g => x => f(g(x))` |

## Church Numerals

| Operation | Lambda | Scala |
|-----------|--------|-------|
| `n` | `λf. λx. fⁿ x` | `f => x => f applied n times to x` |
| `succ` | `λn. λf. λx. f(n f x)` | `n => f => x => f(n(f)(x))` |
| `add` | `λm. λn. λf. λx. m f (n f x)` | `m => n => f => x => m(f)(n(f)(x))` |
| `mul` | `λm. λn. λf. m(n f)` | `m => n => f => m(n(f))` |
| `pow` | `λm. λn. n m` | `m => n => n(m)` |
| `pred` | Pair trick | `n => n(step)((zero, zero))._1` |
| `isZero` | `λn. n(λ_. false) true` | Check if n applies f 0 times |
| `toInt` | N/A | `n(_ + 1)(0)` — count applications |

### Core Insights

1. **A Church numeral `n` is a function** that applies another function exactly `n` times — it is the abstract notion of iteration
2. **Church booleans are if-then-else** — a boolean and a conditional are the same function
3. **Addition** = compose two chains of applications
4. **Multiplication** = iterate the addition chain (applying a chain `m` times)
5. **Exponentiation** = just `n(m)` — the simplest formula of all
6. **Everything computable** can be expressed with just three lambda constructs and one reduction rule

Lambda calculus demonstrates that **functions are the only primitive** needed for computation — all data, all control flow, all arithmetic are derivable from pure function abstraction and application.
